## Apigee ingestion — driver / config

Fetches Elasticsearch hits per 30-minute interval and appends them to `all_hits`.

**Change:** for a daily run, it ingests **yesterday's data** (the last completed day). `end_date`/`start_date` are computed dynamically as `today - 1 day` in the Asia/Kolkata timezone, so no date needs editing each day.

In [ ]:
import pytz
from datetime import datetime, timedelta

kolkata_tz = pytz.timezone("Asia/Kolkata")

url = "https://10.227.12.188:9201/apigee_updated_solace*/_search"

headers = {
    "Content-Type": "application/vnd.elasticsearch+json; compatible-with=8",
    "Accept": "application/vnd.elasticsearch+json; compatible-with=8",
}

all_scopes = list(SCOPE_JOURNEY_MAP.keys())
print(f"all_scopes : {all_scopes}")
# scope = ['TDCC','SWCC','PPCC','PBCC','PTCC']
# scope = ['PBCC']

# Ingest yesterday's data (last completed day, in Asia/Kolkata)
end_date   = (datetime.now(kolkata_tz) - timedelta(days=1)).date()
start_date = end_date
print(f"ingesting for last day: {end_date}")

interval_minute = 30
total_intervals = (24 * 60) // interval_minute
all_hits = []
current_date = start_date

while current_date <= end_date:
    file_date = current_date.strftime("%Y-%m-%d")
    print(f"processing {file_date}")
    for i in range(total_intervals):
        start_total_min = i * interval_minute
        end_total_min   = start_total_min + interval_minute - 1
        start_hour   = start_total_min // 60
        start_minute = start_total_min % 60
        end_hour     = end_total_min // 60
        end_minute   = end_total_min % 60
        start_time = f"{file_date}T{start_hour:02d}:{start_minute:02d}:00.000"
        # ... existing per-interval Elasticsearch query + all_hits.extend(...) ...
